# Анализ визитов и регистраций

Загрузка данных о визитах и регистрациях из API и предварительный анализ.

In [1]:
import os
from io import StringIO

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
API_URL = os.getenv('API_URL', 'https://data-charts-api.hexlet.app')
DATE_BEGIN = os.getenv('DATE_BEGIN', '2023-03-01')
DATE_END = os.getenv('DATE_END', '2023-09-01')
ADS_URL = os.getenv(
    'ADS_URL',
    'https://drive.google.com/uc?export=download&id=12vCtGhJlcK_CBcs8ES3BfEPbk6OJ45Qj',
)

## Загрузка данных из API

Маршруты `/visits` и `/registrations` принимают `begin` и `end` в формате `YYYY-MM-DD`. Дата `end` в выборку не входит.

In [2]:
def fetch_data(route: str, begin: str, end: str) -> pd.DataFrame:
    """Запрашивает данные из API за период [begin, end) и возвращает DataFrame."""
    response = requests.get(
        f'{API_URL}/{route}',
        params={'begin': begin, 'end': end},
        timeout=120,
    )
    response.raise_for_status()
    df = pd.DataFrame(response.json())
    df['datetime'] = pd.to_datetime(df['datetime'])
    return df


visits = fetch_data('visits', DATE_BEGIN, DATE_END)
registrations = fetch_data('registrations', DATE_BEGIN, DATE_END)
print(f'Визиты: {len(visits)}, регистрации: {len(registrations)}')

Визиты: 263459, регистрации: 21836


## Визиты

In [3]:
visits.info()
visits.head()

<class 'pandas.DataFrame'>
RangeIndex: 263459 entries, 0 to 263458
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   visit_id    263459 non-null  str           
 1   platform    263459 non-null  str           
 2   user_agent  263459 non-null  str           
 3   datetime    263459 non-null  datetime64[us]
dtypes: datetime64[us](1), str(3)
memory usage: 8.0 MB


,visit_id,platform,user_agent,datetime
0,c1f947b0-456f-45f5-b6b3-ffba5a426baf,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,2023-06-28 13:43:28
1,c1f947b0-456f-45f5-b6b3-ffba5a426baf,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,2023-06-26 11:23:23
2,a140803f-a641-4cc2-8e2c-49ac6981bd77,web,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:15....,2023-06-28 14:54:40
3,a140803f-a641-4cc2-8e2c-49ac6981bd77,web,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:15....,2023-06-26 03:36:10
4,a140803f-a641-4cc2-8e2c-49ac6981bd77,web,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:15....,2023-06-26 20:22:14


In [4]:
visits.describe(include='all')

,visit_id,platform,user_agent,datetime
count,263459,263459,263459,263459
unique,146085,4,32,NaN
top,64167edd-323a-4ab0-be9b-acd237a1ac30,web,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109...,NaN
freq,4,236301,13623,NaN
mean,NaN,NaN,NaN,2023-06-01 06:22:22.042044
min,NaN,NaN,NaN,2023-03-01 00:00:43
25%,NaN,NaN,NaN,2023-04-13 19:45:03
50%,NaN,NaN,NaN,2023-06-03 12:23:27
75%,NaN,NaN,NaN,2023-07-17 22:09:08
max,NaN,NaN,NaN,2023-08-31 23:52:57


In [5]:
visits['platform'].value_counts()

platform
web        236301
android     13972
bot          7382
ios          5804
Name: count, dtype: int64

## Регистрации

In [6]:
registrations.info()
registrations.head()

<class 'pandas.DataFrame'>
RangeIndex: 21836 entries, 0 to 21835
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   datetime           21836 non-null  datetime64[us]
 1   user_id            21836 non-null  str           
 2   email              21836 non-null  str           
 3   platform           21836 non-null  str           
 4   registration_type  21836 non-null  str           
dtypes: datetime64[us](1), str(4)
memory usage: 853.1 KB


,datetime,user_id,email,platform,registration_type
0,2023-03-01 07:40:13,2e0f6bb8-b029-4f45-a786-2b53990d37f1,ebyrd@example.org,web,google
1,2023-03-01 13:14:00,f007f97c-9d8b-48b5-af08-119bb8f6d9b6,knightgerald@example.org,web,email
2,2023-03-01 03:05:50,24ff46ae-32b3-4a74-8f27-7cf0b8f32f15,cherylthompson@example.com,web,apple
3,2023-03-01 00:04:47,3e9914e1-5d73-4c23-b25d-b59a3aeb2b60,halldavid@example.org,web,email
4,2023-03-01 18:31:52,27f875fc-f8ce-4aeb-8722-0ecb283d0760,denise86@example.net,web,google


In [7]:
registrations.describe(include='all')

,datetime,user_id,email,platform,registration_type
count,21836,21836,21836,21836,21836
unique,NaN,21836,20868,3,4
top,NaN,2e0f6bb8-b029-4f45-a786-2b53990d37f1,ujones@example.com,android,email
freq,NaN,1,6,10582,8996
mean,2023-05-29 10:17:02.002747,NaN,NaN,NaN,NaN
min,2023-03-01 00:04:47,NaN,NaN,NaN,NaN
25%,2023-04-12 01:49:47.500000,NaN,NaN,NaN,NaN
50%,2023-05-26 13:22:09,NaN,NaN,NaN,NaN
75%,2023-07-15 17:26:30.500000,NaN,NaN,NaN,NaN
max,2023-08-31 23:43:26,NaN,NaN,NaN,NaN


In [8]:
registrations[['platform', 'registration_type']].value_counts()

platform  registration_type
android   google               5388
          email                5194
ios       apple                2262
          email                2115
web       apple                1744
          yandex               1729
          google               1717
          email                1687
Name: count, dtype: int64

## Конверсия визитов в регистрации

1. Убираем визиты ботов: поисковые и SEO-боты определяются по слову `bot` в User-Agent.
2. Пользователь мог заходить несколько раз до регистрации, поэтому для каждого `visit_id` берём только последний визит.
3. Группируем визиты и регистрации по дате и платформе, объединяем и считаем конверсию в процентах.

In [9]:
is_bot = visits['user_agent'].str.contains('bot', case=False)
print(f'Визитов ботов: {is_bot.sum()}')

last_visits = (
    visits[~is_bot]
    .sort_values('datetime')
    .drop_duplicates(subset='visit_id', keep='last')
)
print(f'Визитов без ботов, последний визит на visit_id: {len(last_visits)}')

Визитов ботов: 7382
Визитов без ботов, последний визит на visit_id: 138703


In [10]:
visits_by_day = (
    last_visits
    .assign(date_group=last_visits['datetime'].dt.normalize())
    .groupby(['date_group', 'platform'])
    .size()
    .reset_index(name='visits')
)

registrations_by_day = (
    registrations
    .assign(date_group=registrations['datetime'].dt.normalize())
    .groupby(['date_group', 'platform'])
    .size()
    .reset_index(name='registrations')
)

In [11]:
conversion = visits_by_day.merge(
    registrations_by_day, on=['date_group', 'platform'], how='outer'
)
conversion[['visits', 'registrations']] = (
    conversion[['visits', 'registrations']].fillna(0).astype(int)
)
conversion['conversion'] = conversion['registrations'] / conversion['visits'] * 100
conversion = conversion.sort_values(['date_group', 'platform']).reset_index(drop=True)
conversion.head(10)

,date_group,platform,visits,registrations,conversion
0,2023-03-01,android,75,61,81.333333
1,2023-03-01,ios,22,18,81.818182
2,2023-03-01,web,279,8,2.867384
3,2023-03-02,android,67,59,88.059701
4,2023-03-02,ios,31,24,77.419355
5,2023-03-02,web,515,23,4.466019
6,2023-03-03,android,26,22,84.615385
7,2023-03-03,ios,40,34,85.000000
8,2023-03-03,web,617,51,8.265802
9,2023-03-04,android,94,77,81.914894


In [12]:
conversion.describe()

,date_group,visits,registrations,conversion
count,552,552.000000,552.000000,552.000000
mean,2023-05-31 12:00:00,251.273551,39.557971,52.835917
min,2023-03-01 00:00:00,7.000000,6.000000,2.251407
25%,2023-04-15 18:00:00,36.000000,23.000000,7.388262
50%,2023-05-31 12:00:00,74.500000,34.000000,70.479303
75%,2023-07-16 06:00:00,473.250000,54.000000,80.676179
max,2023-08-31 00:00:00,1295.000000,114.000000,96.153846
std,NaN,309.435871,22.078237,34.093905


In [13]:
# Дата сохраняется как Unix-время в миллисекундах (формат из примера задания)
epoch_ms = (conversion['date_group'] - pd.Timestamp(0)) // pd.Timedelta(milliseconds=1)
conversion.assign(date_group=epoch_ms).to_json('./conversion.json')

## Рекламные кампании

Загружаем CSV с затратами на рекламу и присоединяем к визитам и регистрациям по дате. Для дней без рекламы `cost = 0`, `utm_campaign = 'none'`.

In [14]:
response = requests.get(ADS_URL, timeout=60)
response.raise_for_status()
ads = pd.read_csv(StringIO(response.text), parse_dates=['date'])
ads.info()
ads.head()

<class 'pandas.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          159 non-null    datetime64[us]
 1   utm_source    159 non-null    str           
 2   utm_medium    159 non-null    str           
 3   utm_campaign  159 non-null    str           
 4   cost          159 non-null    int64         
dtypes: datetime64[us](1), int64(1), str(3)
memory usage: 6.3 KB


,date,utm_source,utm_medium,utm_campaign,cost
0,2023-03-01 10:54:41,google,cpc,advanced_algorithms_series,212
1,2023-03-02 10:32:35,google,cpc,advanced_algorithms_series,252
2,2023-03-03 19:21:40,google,cpc,advanced_algorithms_series,202
3,2023-03-04 17:52:04,google,cpc,advanced_algorithms_series,223
4,2023-03-05 05:35:13,google,cpc,advanced_algorithms_series,265


In [15]:
ads.describe(include='all')

,date,utm_source,utm_medium,utm_campaign,cost
count,159,159,159,159,159.000000
unique,NaN,3,2,5,NaN
top,NaN,google,cpc,virtual_reality_workshop,NaN
freq,NaN,73,116,43,NaN
mean,2023-06-12 10:36:50.157232,NaN,NaN,NaN,194.836478
min,2023-03-01 10:54:41,NaN,NaN,NaN,100.000000
25%,2023-04-19 09:19:35.500000,NaN,NaN,NaN,143.000000
50%,2023-06-09 15:22:47,NaN,NaN,NaN,190.000000
75%,2023-08-08 19:59:32,NaN,NaN,NaN,252.000000
max,2023-09-17 02:04:39,NaN,NaN,NaN,298.000000


In [16]:
ads_by_day = (
    ads
    .assign(date_group=ads['date'].dt.normalize())
    .groupby('date_group')
    .agg(cost=('cost', 'sum'), utm_campaign=('utm_campaign', 'first'))
    .reset_index()
)

totals_by_day = (
    conversion
    .groupby('date_group')[['visits', 'registrations']]
    .sum()
    .reset_index()
)

ads_conversion = totals_by_day.merge(ads_by_day, on='date_group', how='left')
ads_conversion['cost'] = ads_conversion['cost'].fillna(0).astype(int)
ads_conversion['utm_campaign'] = ads_conversion['utm_campaign'].fillna('none')
ads_conversion = ads_conversion.sort_values('date_group').reset_index(drop=True)
ads_conversion.head(10)

,date_group,visits,registrations,cost,utm_campaign
0,2023-03-01,376,87,212,advanced_algorithms_series
1,2023-03-02,613,106,252,advanced_algorithms_series
2,2023-03-03,683,107,202,advanced_algorithms_series
3,2023-03-04,647,159,223,advanced_algorithms_series
4,2023-03-05,707,115,265,advanced_algorithms_series
5,2023-03-06,1291,230,108,advanced_algorithms_series
6,2023-03-07,1382,124,165,advanced_algorithms_series
7,2023-03-08,1382,151,155,advanced_algorithms_series
8,2023-03-09,1064,209,124,advanced_algorithms_series
9,2023-03-10,812,112,276,advanced_algorithms_series


In [17]:
ads_conversion.groupby('utm_campaign').agg(
    start=('date_group', 'min'),
    end=('date_group', 'max'),
    days=('date_group', 'count'),
    visits=('visits', 'sum'),
    registrations=('registrations', 'sum'),
    cost=('cost', 'sum'),
).sort_values('start')

,start,end,days,visits,registrations,cost
utm_campaign,,,,,,
advanced_algorithms_series,2023-03-01,2023-03-15,15,14706,2196,2824
none,2023-03-16,2023-08-05,42,22956,4004,0
virtual_reality_workshop,2023-03-25,2023-05-06,43,32698,5898,9031
ui_ux_design_drive,2023-05-19,2023-06-09,22,14398,2308,4209
women_in_tech_symposium,2023-06-21,2023-07-26,36,31028,4184,6780
intro_to_python_course,2023-08-06,2023-08-31,26,22917,3246,4690


In [18]:
epoch_ms = (ads_conversion['date_group'] - pd.Timestamp(0)) // pd.Timedelta(milliseconds=1)
ads_conversion.assign(date_group=epoch_ms).to_json('./ads.json')